# IRRM-CODEC inverse: run and analyze

This notebook mirrors the forward example, but trains and evaluates the inverse model that reconstructs a CDR3 sequence from a TCRemP embedding.

## 1. Define inputs

The AIRR file must contain `clone_id`, `junction_aa`, `v_call`, `j_call`, `locus`.
The embeddings parquet must contain `clone_id` plus either `tcremp_emb` or numeric embedding columns.

In the model pipeline, CDR3 sequences are gap-padded to a fixed length of 40 amino acids before tokenization.

In [ ]:
from pathlib import Path

from rtp_codec.paths import repository_root
root = repository_root()
airr_path = '/projects/immunestatus/vdjdb_olga/airr_format/trb_background_100k.tsv'  # root / 'data' / 'sample_airr.tsv'
embeddings_path = '/projects/immunestatus/vdjdb_olga/tcremp/trb_background_100k_embeddings.parquet'
output_dir = root / 'artifacts' / 'inverse_demo_trb'

airr_path, embeddings_path, output_dir

In [ ]:
import sys

sys.path.insert(0, str(root / "src"))

## 2. Preview the two files

In [ ]:
import pandas as pd

airr_df = pd.read_csv(airr_path, sep='\t')
emb_df = pd.read_parquet(embeddings_path)

display(airr_df.head())
display(emb_df.head())

## 3. Launch training

In [ ]:
import subprocess

cmd = [
    sys.executable, '-m', 'rtp_codec.training.inverse',
    '--airr-path', str(airr_path),
    '--embeddings-path', str(embeddings_path),
    '--output-dir', str(output_dir),
    '--locus', 'beta',
    '--epochs', '25',
    '--max-len', '40',
    '--weight-decay', '0.00001',
    '--lr', '0.001'
]
subprocess.run(cmd, cwd=root, check=True)
cmd

## 4. Inspect saved metrics and merge stats

In [ ]:
import json
import pandas as pd

history = json.loads((output_dir / 'history.json').read_text())
test_metrics = json.loads((output_dir / 'test_metrics.json').read_text())
data_stats = json.loads((output_dir / 'data_stats.json').read_text())
history_df = pd.json_normalize(history)

display(history_df)
display(test_metrics)
display(data_stats)

In [ ]:
history_df.plot(x='epoch', y=['train.loss', 'val.loss'], figsize=(8, 4), grid=True, title='Loss by epoch')

In [ ]:
history_df.plot(
    x='epoch',
    y=['val.token_accuracy', 'val.length_accuracy', 'val.exact_match'],
    figsize=(8, 4),
    grid=True,
    title='Validation metrics by epoch',
)

## 5. Load a checkpoint

In [ ]:
import torch
from rtp_codec.models.inverse import InverseModel

checkpoint = torch.load(output_dir / 'best.pt', map_location='cpu')
extra = checkpoint['extra']
model = InverseModel(
    embedding_dim=extra['embedding_dim'],
    max_len=extra.get('max_len', 40),
)
model.load_state_dict(checkpoint['model_state'])
model.eval()
checkpoint['metrics']

## 6. Test the model on held-out examples

In [ ]:
import numpy as np
import pandas as pd
import torch

from rtp_codec.data.io import load_airr_with_embeddings
from rtp_codec.tokenization.character import decode
from rtp_codec.utils import apply_standardizer, split_indices

df, emb, merge_stats = load_airr_with_embeddings(
    airr_path=airr_path,
    embeddings_path=embeddings_path,
    locus="beta",
)

train_idx, val_idx, test_idx = split_indices(
    len(df),
    train_fraction=0.8,
    val_fraction=0.1,
    seed=42,
)

test_df = df.iloc[test_idx].reset_index(drop=True)
test_emb = apply_standardizer(
    emb[test_idx],
    np.load(output_dir / "mean.npy"),
    np.load(output_dir / "std.npy"),
)

sample_size = 5000
sample_emb = torch.tensor(test_emb[:sample_size], dtype=torch.float32)

with torch.no_grad():
    pred_tokens = model.generate(sample_emb, max_len=model.max_len)

predicted_aligned = [decode(row.tolist()) for row in pred_tokens]
predicted = [decode(row.tolist(), remove_gaps=True) for row in pred_tokens]

results = pd.DataFrame({
    "target": test_df["junction_aa"].iloc[:sample_size].tolist(),
    "predicted_aligned": predicted_aligned,
    "predicted": predicted,
})

results["target_length"] = results["target"].str.len()
results["predicted_length"] = results["predicted"].str.len()
results["exact_match"] = results["target"] == results["predicted"]
results["length_match"] = results["target_length"] == results["predicted_length"]

results


In [ ]:
results.exact_match.mean()

In [ ]:
results.length_match.mean()

In [ ]:
results[~results['exact_match']]

In [ ]:
results[~results.length_match]

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

aa_order = list("CSTAGPDEQNHRKMILVWYF")

groups = {
    "C": set("C"),
    "STAGP": set("STAGP"),
    "DEQN": set("DEQN"),
    "HRK": set("HRK"),
    "MILV": set("MILV"),
    "WYF": set("WYF"),
}

aa_to_group = {}
for group_name, aas in groups.items():
    for aa in aas:
        aa_to_group[aa] = group_name

# берём только случаи, где длины после удаления гэпов совпадают
df_sub = results[results["length_match"]].copy()

conf_mat = pd.DataFrame(0, index=aa_order, columns=aa_order, dtype=int)

within_group_errors = 0
between_group_errors = 0
total_errors = 0

for target_seq, pred_seq in zip(df_sub["target"], df_sub["predicted"]):
    if len(target_seq) != len(pred_seq):
        continue

    for t_aa, p_aa in zip(target_seq, pred_seq):
        if t_aa not in aa_order or p_aa not in aa_order:
            continue
        if t_aa == p_aa:
            continue

        conf_mat.loc[t_aa, p_aa] += 1
        total_errors += 1

        if aa_to_group[t_aa] == aa_to_group[p_aa]:
            within_group_errors += 1
        else:
            between_group_errors += 1

plt.figure(figsize=(11, 9))
ax = sns.heatmap(
    conf_mat,
    annot=True,
    fmt="d",
    cmap="Greys",
    cbar=True,
    square=True,
    linewidths=0.5,
    linecolor="lightgray",
)

# только границы групп, без цветной заливки
boundaries = np.cumsum([1, 5, 4, 3, 4])  # C | STAGP | DEQN | HRK | MILV | WYF
for b in boundaries:
    ax.hlines(b, 0, len(aa_order), colors="black", linewidth=2)
    ax.vlines(b, 0, len(aa_order), colors="black", linewidth=2)

ax.set_xlabel("Predicted amino acid")
ax.set_ylabel("Target amino acid")
ax.set_title("Substitution confusion matrix")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    "count": [within_group_errors, between_group_errors, total_errors],
    "fraction": [
        within_group_errors / total_errors if total_errors else np.nan,
        between_group_errors / total_errors if total_errors else np.nan,
        1.0 if total_errors else np.nan,
    ],
}, index=["within_group", "between_group", "total"])

display(summary)
